In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [2]:
model_name = "gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 82.00 MiB. GPU 0 has a total capacity of 23.64 GiB of which 11.00 MiB is free. Process 247341 has 20.70 GiB memory in use. Including non-PyTorch memory, this process has 2.92 GiB memory in use. Of the allocated memory 2.54 GiB is allocated by PyTorch, and 5.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Use SHAP for token weights

In [ ]:
explainer = shap.Explainer(model, tokenizer)

In [ ]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [ ]:
shap_values = explainer(prompts)

In [ ]:
print(shap_values.values.squeeze())

[-3.50681455e-06  2.76400235e+00 -8.72354661e-01  2.00342429e+00
  4.72185472e+00  1.18334559e+00  1.11659276e-01  2.30677287e+00
  9.67523081e-01  5.83311160e+00  2.01544604e+00]


In [ ]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze())
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [ ]:
# For now just do it on the web UI since we don't have a dataset yet

## ASG Pipeline

### Prune

In [3]:
json_path = "temp_graph_files/austin.json"
source_set = 'clt-hp' #'clt-hp' # gemmascope-transcoder-16k
# token_weights = [0.00198786, 0.03153391, 0.00083086, 0.01473883, 0.22338926, 0.00649094,
#  0.00222269, 0.01996207, 0.0052309, 0.67869559, 0.01491708]
token_weights = [0, 0, 0, 0, 1/3, 0, 0, 1/3, 0, 1/3, 0]
kept_ids, pruned_adj, node_inf, node_rel, attr, metadata = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.7,
    edge_influence_threshold=0.9,
    node_relevance_threshold=0.8,
    edge_relevance_threshold=0.9,
    keep_all_tokens_and_logits=False,
)

In [4]:
print(len(kept_ids))

25


In [5]:
for i, node_id in enumerate(kept_ids):
    print(node_id, attr[node_id].get("clerp", ""), node_inf[i].item(), node_rel[i].item())

1_89326_9 Dallas 0.6338451504707336 0.6352778077125549
7_24743_9 Texas legal matters 0.6120493412017822 0.6976507902145386
8_21080_9 Texas legal contexts 0.6929287910461426 0.7583836913108826
16_89970_9 Texas 0.5723963379859924 0.6424320340156555
16_98048_10 cities 0.6091805696487427 0.723612904548645
17_1822_10 place names 0.6403317451477051 0.5670413374900818
17_98126_10 Government buildings/institutions 0.6271374821662903 0.6482846140861511
18_3623_10 capital cities 0.5764042735099792 0.6277174353599548
18_45367_10 US cities 0.6881327033042908 0.5960058569908142
18_61980_10 Texas related 0.6503652334213257 0.6506885290145874
19_54790_10 Places 0.6360085010528564 0.6441317796707153
20_44686_9 Texas locations/legal contexts 0.5682745575904846 0.5120062232017517
20_44686_10 Texas locations/legal contexts 0.5273332595825195 0.5191220045089722
20_74108_10 capital 0.5873730778694153 0.5618082284927368
21_16875_10 cities 0.6174425482749939 0.6015009880065918
21_61721_10 code/documentation 

### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [25]:
feature_types = classify_features(kept_ids, attr, metadata)
print(feature_types)

IndexError: list index out of range

# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold
2. Initialize with Shap (normalize)
3. Normalize adjacency matrix

## Grouping
1. Classify: Classify and relabel using LLM
2. Group: Group using LLM

## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality